In [ ]:
import os
os.chdir('/kaggle/working')

!git clone https://github.com/basiaseweryn/generative-models-cat-images.git

In [ ]:
%cd /kaggle/working/generative-models-cat-images

In [ ]:
!git pull 

In [ ]:
import sys
import os
import os


PROJECT_ROOT = "/kaggle/working/generative-models-cat-images" 

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

UTILS_PATH = os.path.join(PROJECT_ROOT, "utils")
if UTILS_PATH not in sys.path:
    sys.path.insert(0, UTILS_PATH)

print(f"project root added: {PROJECT_ROOT}")
print(f"utils folder added: {UTILS_PATH}")
print(f"available files: {os.listdir(PROJECT_ROOT)}")

In [ ]:
import os
import sys
import torch

CATS_DATASET = "/kaggle/input/datasets/crawford/cat-dataset"
OUTPUT_DIR = "/kaggle/working/generative-models-cat-images/outputs"
PROJECT_DIR = "/kaggle/working/generative-models-cat-images"
FID_SAMPLES = 2000  

sys.path.insert(0, PROJECT_DIR)

import utils.config as cfg
cfg.DATA_DIR = CATS_DATASET
cfg.NUM_WORKERS = 0
cfg.OUTPUT_DIR = OUTPUT_DIR + "/"

from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from utils.config import STAGE_1_SCENARIOS, STAGE_2_SCENARIOS, seed_everything
from utils.evaluation_utils import (
    compare_fid_scenarios,
    generate_candidate_grid,
    interpolate_between_latents,
    load_trained_model,
    save_interpolation_grid,
    save_latent_codes,
    save_sample_grid,
    show_image_grid,
)


def list_result_scenarios(results_dir):
    root = Path(results_dir)
    return sorted(p.name for p in root.iterdir() if p.is_dir() and (p / "model.pth").exists())


def list_sample_checkpoints(scenario, results_dir):
    paths = sorted(
        (Path(results_dir) / scenario).glob("samples_epoch_*.png"),
        key=lambda p: int(p.stem.split("_")[-1]),
    )
    return [str(p) for p in paths]


def show_checkpoint_progression(scenario, results_dir):
    paths = list_sample_checkpoints(scenario, results_dir)
    if not paths:
        print("No sample PNGs for", scenario)
        return
    fig, axes = plt.subplots(1, len(paths), figsize=(4 * len(paths), 4))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(Image.open(p))
        ax.set_title(Path(p).name.replace(".png", ""))
        ax.axis("off")
    fig.suptitle(f"Training progression — {scenario}")
    plt.tight_layout()
    plt.show()


def show_saved_image(path, title=""):
    plt.figure(figsize=(10, 10))
    plt.imshow(Image.open(path))
    plt.title(title)
    plt.axis("off")
    plt.show()


def compute_fid_report(names, device, results_dir, max_samples, output_csv, label):
    available = [s for s in names if (Path(results_dir) / s / "model.pth").exists()]
    if not available:
        print(f"[{label}] no checkpoints found")
        return []
    print(f"[{label}] Computing FID:", available)
    return compare_fid_scenarios(
        available, device, max_samples=max_samples,
        models_dir=results_dir, save_csv=output_csv,
    )


def recommend_for_stage3(stage1_rows, stage2_rows):
    rec = {"vae_for_interpolation": "stage_1_vae", "dcgan_for_interpolation": "stage_1_dcgan", "notes": []}
    for rows in (stage1_rows, stage2_rows):
        if not rows:
            continue
        for model in ("vae", "dcgan"):
            sub = [r for r in rows if r.get("model") == model]
            if sub:
                best = min(sub, key=lambda r: float(r["fid"]))["scenario"]
                if rows is stage1_rows:
                    rec[f"{model}_for_interpolation"] = best
                else:
                    rec[f"best_{model}_stage2"] = best
    rec["notes"].append("Use stage_1_* for interpolation (full data, 50 epochs).")
    return rec


def find_models_dir(project_dir):
    """Folder with stage_*/model.pth (handles trained_models/trained_models/ nesting)."""
    base = os.path.join(project_dir, "trained_models")
    for candidate in [base, os.path.join(base, "trained_models")]:
        if list_result_scenarios(candidate):
            return candidate
    return base


MODELS_DIR = find_models_dir(PROJECT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything()
os.makedirs(f"{OUTPUT_DIR}/reports", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/stage_3", exist_ok=True)

print("Device:", device)
print("Cats data:", CATS_DATASET, "exists:", os.path.isdir(CATS_DATASET))
print("Models dir:", MODELS_DIR)
print("Checkpoints:", list_result_scenarios(MODELS_DIR))

## Stage 1 — Baseline (VAE vs DCGAN)

- Training sample grids (every 10 epochs)  
- **FID** — lower is better  
- Final samples at epoch 50

In [ ]:
for name in STAGE_1_SCENARIOS:
    if name in list_result_scenarios(MODELS_DIR):
        show_checkpoint_progression(name, MODELS_DIR)

In [ ]:
fid_stage1 = compute_fid_report(
    STAGE_1_SCENARIOS,
    device,
    results_dir=MODELS_DIR,
    max_samples=FID_SAMPLES,
    output_csv=f"{OUTPUT_DIR}/reports/fid_stage1.csv",
    label="stage_1",
)

In [ ]:
for name in STAGE_1_SCENARIOS:
    paths = list_sample_checkpoints(name, MODELS_DIR)
    if paths:
        show_saved_image(paths[-1], title=f"Final samples — {name}")

## Stage 2 — Hyperparameters

These runs used **25% of the data** (`subset_ratio=0.25`). Compare FID **within stage 2** only.  
For interpolation (stage 3), prefer **stage 1** models (full data, 50 epochs) unless you retrained the winner on 100% data.

In [ ]:
stage2_available = [s for s in STAGE_2_SCENARIOS if s in list_result_scenarios(MODELS_DIR)]
print("Stage 2 checkpoints:", stage2_available)

for name in stage2_available:
    show_checkpoint_progression(name, MODELS_DIR)

In [ ]:
fid_stage2 = []
if stage2_available:
    fid_stage2 = compute_fid_report(
        stage2_available,
        device,
        results_dir=MODELS_DIR,
        max_samples=FID_SAMPLES,
        output_csv=f"{OUTPUT_DIR}/reports/fid_stage2_hyperparams.csv",
        label="stage_2",
    )

In [ ]:
rec = recommend_for_stage3(fid_stage1 or [], fid_stage2 or [])
print("\n=== Suggestion for Stage 3 ===")
print("VAE scenario:  ", rec["vae_for_interpolation"])
print("DCGAN scenario:", rec["dcgan_for_interpolation"])
for note in rec["notes"]:
    print(" -", note)

## Stage 3 — Latent interpolation (10 images)

1. Run the grid cell and **pick two image indices**  
2. Set `IDX_A` and `IDX_B` in the next cell  
3. You get **2 endpoints + 8 intermediate** generations (linear interpolation in latent space)

For the report: comment on whether the transition is smooth (structured latent space).

In [ ]:
# Pick one model — usually stage_1_vae or stage_1_dcgan from the suggestion above
SCENARIO = rec.get("vae_for_interpolation", "stage_1_vae")
# SCENARIO = rec.get("dcgan_for_interpolation", "stage_1_dcgan")

model, config, _ = load_trained_model(SCENARIO, device, models_dir=MODELS_DIR)
display_imgs, latents, _ = generate_candidate_grid(
    model, config, device, num_candidates=64, seed=123
)
show_image_grid(display_imgs, title=f"Pick two images (0–63) — {SCENARIO}", nrow=8)
save_sample_grid(
    display_imgs,
    f"{OUTPUT_DIR}/stage_3/{SCENARIO}_candidate_grid.png",
    nrow=8,
)

In [ ]:
# Change after looking at the grid (row-major, 8 columns)
IDX_A = 3
IDX_B = 47

z_a = latents[IDX_A].clone()
z_b = latents[IDX_B].clone()

latent_path = f"{OUTPUT_DIR}/stage_3/{SCENARIO}_interpolation_latents.pth"
save_latent_codes(z_a, z_b, latent_path)
print("Saved latent vectors:", latent_path)
print("z shape:", z_a.shape)

In [ ]:
interp_display, interp_raw, is_dcgan = interpolate_between_latents(
    model, config, z_a, z_b, device, num_steps=8
)

interp_path = f"{OUTPUT_DIR}/stage_3/{SCENARIO}_interpolation_10.png"
save_interpolation_grid(interp_raw, interp_path, is_dcgan=is_dcgan)
show_image_grid(
    interp_display,
    title=f"Interpolation {IDX_A} → {IDX_B} ({SCENARIO})",
    nrow=10,
)

# Endpoints only (the two selected generations)
endpoints = torch.stack([interp_display[0], interp_display[-1]])
show_image_grid(endpoints, title="Endpoints", nrow=2)

In [ ]:
# Optional: repeat stage 3 for DCGAN (uncomment)
# SCENARIO = "stage_1_dcgan"
# model, config, _ = load_trained_model(SCENARIO, device, models_dir=MODELS_DIR)
# ...

## Download results

Click **Save Version**, then download from **Output**:

| File | Description |
|------|-------------|
| `outputs/reports/fid_stage1.csv` | VAE vs DCGAN FID |
| `outputs/reports/fid_stage2_hyperparams.csv` | Hyperparameter comparison |
| `outputs/stage_3/*` | Interpolation PNG + `.pth` latents |

In [ ]:
!find {OUTPUT_DIR} -type f | head -30